# 06c | Drills: Statistical inference

Goes with **06a and 06b**: confidence intervals, the various t-tests, proportion tests, chi-square tests, nonparametric tests, bootstrap, permutation tests, power simulation, multiple testing, MLE.

**How to use this page:** There are no input boxes here. There are two kinds of problems:

- **Computation problems**: the code and its output are shown right below. First think about which function and formula to use, then read on for the answer;
- **Spoken problems** (marked 🗣 after the title): answer out loud first (30–60 seconds), then compare with the reference answer.

Problems on a page run in order, so variables and functions defined earlier can be reused later. Every problem assumes this has already been run:

```python
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(2026)
x = rng.normal(52, 10, 30)                        # one group of 30 observations
before = rng.normal(80, 12, 25)                   # 25 people before the intervention
after = before - 3 + rng.normal(0, 5, 25)         # the same people after the intervention
```

### Problem 1 · 95% CI for a mean, by hand

The 95% confidence interval for the mean of `x` (without a built-in function).

```python
n = len(x)
se = x.std(ddof=1) / np.sqrt(n)
tcrit = stats.t.ppf(0.975, df=n - 1)
print(round(x.mean(), 3), round(x.mean() - tcrit * se, 3), round(x.mean() + tcrit * se, 3))
```

**Result:**

```text
51.969 48.951 54.988
```

> Sample standard deviation with `ddof=1`; the critical value comes from the t distribution with n − 1 degrees of freedom.

### Problem 2 · One-sample t-test

Test whether the population mean of `x` equals 50: compute t and p by hand first, then check with `stats.ttest_1samp`.

```python
t_stat = (x.mean() - 50) / se
p_hand = 2 * stats.t.sf(abs(t_stat), df=n - 1)
print(round(t_stat, 4), round(p_hand, 4))
res = stats.ttest_1samp(x, 50)
print(round(res.statistic, 4), round(res.pvalue, 4))
```

**Result:**

```text
1.3344 0.1925
1.3344 0.1925
```

> The 95% interval from the previous problem contains 50, so the p-value here must be above 0.05: a two-sided test and a confidence interval are two ways of saying the same thing.

### Problem 3 · Paired t-test

The same people are measured before and after an intervention; test whether anything changed. Then see what happens to the p-value if you wrongly treat them as two independent samples.

```python
print(round(stats.ttest_rel(before, after).pvalue, 6))
print(round(stats.ttest_ind(before, after, equal_var=False).pvalue, 4))
```

**Result:**

```text
0.000989
0.3645
```

> The paired test works with each person's difference, which cancels out the large differences between individuals, so it is far more sensitive. Using the wrong test turns a significant effect into "not significant".

### Problem 4 · Two-proportion z-test

Control: 120 conversions out of 1000; treatment: 150 out of 1000. Test with `proportions_ztest` and check against your hand-written version from 06b.

```python
z, p = proportions_ztest([150, 120], [1000, 1000])
print(round(z, 4), round(p, 4))
```

**Result:**

```text
1.963 0.0496
```

> The first argument is the number of successes in each group, the second the group sizes; the sign of z is "first group − second group". The result matches the hand-written formula from 06b (standard error from the pooled proportion): z ≈ 1.963, p ≈ 0.0496.

### Problem 5 · 2×2 chi-square test

100 new and 100 returning users, of whom 30 and 45 made a purchase. Use a chi-square test to see whether purchase rate is associated with being new or returning.

```python
table = [[30, 70],
         [45, 55]]
chi2, p, dof, expected = stats.chi2_contingency(table)
print(round(chi2, 4), round(p, 4), dof)

chi2_nc, p_nc, _, _ = stats.chi2_contingency(table, correction=False)
z, p_z = proportions_ztest([30, 45], [100, 100])
print(round(chi2_nc, 4), round(z ** 2, 4), round(p_nc, 4), round(p_z, 4))
```

**Result:**

```text
4.1813 0.0409 1
4.8 4.8 0.0285 0.0285
```

> For a 2×2 table `chi2_contingency` **applies a continuity correction by default** (Yates), which makes the p-value conservative. With the correction off, the chi-square statistic equals exactly the z² of the two-proportion z-test, and the p-values match.

### Problem 6 · Skewed data: Mann–Whitney

Waiting times in both groups are right-skewed (exponential). Compare Welch's t-test with the Mann–Whitney U test.

```python
rng = np.random.default_rng(3)
wait_a = rng.exponential(10, 40)
wait_b = rng.exponential(16, 40)
print(round(stats.ttest_ind(wait_b, wait_a, equal_var=False).pvalue, 4))
print(round(stats.mannwhitneyu(wait_b, wait_a).pvalue, 4))
```

**Result:**

```text
0.0039
0.0186
```

> Mann–Whitney (R's `wilcox.test`) uses only ranks and is not affected by extreme values. What it tests is "do values in one group tend to be larger than in the other", which is strictly speaking not equality of means.

### Problem 7 · Bootstrap: CI for the median

95% bootstrap confidence interval for the median of `x` (2000 resamples).

```python
rng = np.random.default_rng(4)
boot = []
for _ in range(2000):
    sample = rng.choice(x, size=len(x), replace=True)
    boot.append(np.median(sample))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(round(np.median(x), 3), round(lo, 3), round(hi, 3))
```

**Result:**

```text
50.92 49.201 54.722
```

> With replacement, n draws each time, take the 2.5% and 97.5% quantiles. The median has no simple standard-error formula, which is exactly where the bootstrap shines.

### Problem 8 · Permutation test: small samples

Two groups of 6 people each: `g1 = [12, 15, 9, 14, 11, 13]`, `g2 = [16, 18, 14, 19, 15, 17]`. Use a permutation test to get the two-sided p-value for the difference in means, and compare with Welch's t-test.

```python
g1 = np.array([12, 15, 9, 14, 11, 13])
g2 = np.array([16, 18, 14, 19, 15, 17])
observed = g2.mean() - g1.mean()
pooled = np.concatenate([g1, g2])

rng = np.random.default_rng(5)
count = 0
n_perm = 10000
for _ in range(n_perm):
    shuffled = rng.permutation(pooled)
    diff = shuffled[6:].mean() - shuffled[:6].mean()
    if abs(diff) >= abs(observed):
        count += 1
print(round(observed, 3), count / n_perm)
print(round(stats.ttest_ind(g2, g1, equal_var=False).pvalue, 4))
```

**Result:**

```text
4.167 0.0127
0.0052
```

> Shuffle labels → distribution of the difference under H₀ → see how extreme the observed difference is. No normality assumption needed, which is especially useful with small samples.

### Problem 9 · Simulating power

30 people per group, a true effect of 0.5 standard deviations, two-sided α = 0.05: what is the power of Welch's t-test?

```python
rng = np.random.default_rng(6)
n_sim, n_per = 5000, 30
control = rng.normal(0.0, 1.0, size=(n_sim, n_per))
treat = rng.normal(0.5, 1.0, size=(n_sim, n_per))
pvals = stats.ttest_ind(treat, control, axis=1, equal_var=False).pvalue
round((pvals < 0.05).mean(), 3)
```

**Result:**

```text
0.484
```

> Less than half. Normal approximation: $\Phi(0.5\sqrt{30/2} - 1.96) \approx \Phi(-0.02) \approx 0.49$; the t-test is slightly lower. For 80% power you need about 64 people per group.

### Problem 10 · Simulating the Type I error rate

Same setup, but the two groups have no true difference. What share should come out significant?

```python
treat0 = rng.normal(0.0, 1.0, size=(n_sim, n_per))
pvals0 = stats.ttest_ind(treat0, control, axis=1, equal_var=False).pvalue
round((pvals0 < 0.05).mean(), 3)
```

**Result:**

```text
0.048
```

> Close to 0.05. This is the standard way to check whether a testing procedure is trustworthy: when there is no effect, its Type I error rate should equal α.

### Problem 11 · Multiple-testing correction

Below are the p-values for 8 metrics. How many are rejected with no correction, with Bonferroni, and with BH? If none of the 8 has a true effect and the tests are independent, what is the probability of at least one false positive?

```python
pvals = [0.002, 0.009, 0.014, 0.021, 0.04, 0.06, 0.3, 0.7]
print(sum(p < 0.05 for p in pvals))
print(multipletests(pvals, alpha=0.05, method="bonferroni")[0].sum())
print(multipletests(pvals, alpha=0.05, method="fdr_bh")[0].sum())
print(round(1 - 0.95 ** 8, 4))
```

**Result:**

```text
5
1
4
0.3366
```

> The Bonferroni threshold is 0.05/8 ≈ 0.00625; BH looks for the largest i with $p_{(i)} \le i/m \cdot \alpha$: the 4th value 0.021 ≤ 0.025, so the first 4 are rejected.

### Problem 12 · MLE: finding the maximum numerically

50 Poisson observations. Compute the log-likelihood over a grid of candidate λ values, find the λ that maximizes it, and compare with the sample mean.

```python
data = stats.poisson.rvs(3.2, size=50, random_state=7)
lams = np.linspace(1, 6, 501)
loglik = [np.log(stats.poisson.pmf(data, lam)).sum() for lam in lams]
print(round(lams[np.argmax(loglik)], 2), round(data.mean(), 2))
```

**Result:**

```text
3.18 3.18
```

> The Poisson MLE is the sample mean, and the numerical result agrees with the derivation (grid spacing 0.01). `np.argmax` returns the index of the maximum (R's `which.max`), which is then used to pick the matching λ from `lams`. `.rvs(size=n)` is R's `rpois`; `random_state` fixes the random seed.

### Problem 13 · Intervals and tests 🗣

A 95% confidence interval is (−0.4, 2.1). Is the corresponding two-sided test of H₀: parameter = 0 significant at the 0.05 level? What about H₀: parameter = 3?

**Reference answer:**

The interval contains 0, so H₀: parameter = 0 **cannot** be rejected at the 0.05 level; it does not contain 3, so H₀: parameter = 3 can be rejected.

A 95% confidence interval is exactly "the set of parameter values not rejected by a two-sided test at the 0.05 level". This correspondence is handy in interviews: give the interval first and mention significance along the way; that carries more information than a p-value alone.

*Short version:* A 95% interval is the set of null values that a two-sided 5% test would not reject, so zero is not rejected and three is.

### Problem 14 · What determines the width of an interval 🗣

What determines the width of a confidence interval for a mean? How would you halve the width?

**Reference answer:**

Width $= 2 \times$ critical value $\times\, s/\sqrt n$. So three things determine it: the confidence level (higher → larger critical value → wider), the standard deviation s of the data (larger → wider), and the sample size n (larger → narrower).

To halve the width you need **4 times** the sample size, because the width is inversely proportional to $\sqrt n$. The other route is to reduce s: more precise measurement, a paired design, covariate adjustment.

### Problem 15 · Two types of error: an example 🗣

Use spam filtering to illustrate Type I and Type II errors. Which is worse?

**Reference answer:**

Let H₀ be "this email is legitimate". A **Type I error** (false positive): a legitimate email is flagged as spam, and the user may miss something important; a **Type II error** (false negative): a spam email gets through, and the user sees one more ad.

Here the Type I error is usually worse, so you accept letting a bit more spam through to keep the rate of misflagged legitimate emails low. Which one is worse depends on the business: in disease screening a missed diagnosis (Type II) is worse. Saying "it depends on the costs of the two errors" makes a good answer.

### Problem 16 · Are these statements about p = 0.03 right? 🗣

An experiment gives p = 0.03. True or false: (1) the probability that H₀ is true is 3%; (2) if H₀ were true, the probability of a result this extreme or more extreme is 3%; (3) the effect is large; (4) if you redo it with new data, you are 97% sure it will be significant again.

**Reference answer:**

Only (2) is right.

(1) is wrong: the p-value is computed **assuming** H₀ holds; it is not the probability that H₀ holds. (3) is wrong: the p-value mixes effect size with sample size, and with a large sample a tiny effect can have a tiny p-value. (4) is wrong: the probability that a replication is significant is the power, which is a different thing from the p-value; at p = 0.03 the chance of a significant replication is often only 50–60%.

*Short version:* Only the second statement is right. A p-value is computed assuming the null; it isn't the probability the null is true, it doesn't measure effect size, and it isn't the replication probability.

### Problem 17 · When to use a nonparametric test 🗣

When would you choose Mann–Whitney over a t-test? What does it cost you?

**Reference answer:**

Use Mann–Whitney with small samples that are clearly skewed or have extreme values, with data that are only ordinal (e.g. 1–5 ratings), or when the question is "which is larger" rather than about means.

Costs: (1) if the data really are normal, it has slightly less power than the t-test (though the loss is small); (2) it does not test a difference in means, so it cannot directly answer "how much does average revenue per user differ"; when the business needs the mean, with large samples you still use the t-test (thanks to the CLT) or a bootstrap of the difference in means.

### Problem 18 · When does the bootstrap fail? 🗣

Are there situations where the bootstrap does not work?

**Reference answer:**

Yes. (1) **Extreme-value statistics**: e.g. the sample maximum; resampling can never produce a value larger than the sample maximum, so the interval is badly too narrow; (2) **samples that are too small**: the sample itself does not represent the population distribution; (3) **non-independent data**: time series, multiple records per user; resampling individual rows underestimates the variance, so you need to resample by block or by user (cluster); (4) with heavy tails or infinite variance, the bootstrap of the mean is also unreliable.

*Short version:* It struggles with extremes like the sample maximum, with very small samples, and with dependent data, where you need block or cluster resampling.

### Problem 19 · Why a paired design has more power 🗣

With the same 25 people, why is "measure the same people before and after" more likely to detect an effect than "two groups of 25"?

**Reference answer:**

A paired design analyzes each person's own difference $d_i = \text{after} - \text{before}$. Baseline differences between people (some are high to begin with, some low) cancel out in the difference.

$\mathrm{Var}(\text{after} - \text{before}) = \sigma^2_{\text{before}} + \sigma^2_{\text{after}} - 2\,\mathrm{Cov}$: the stronger the positive correlation between before and after, the smaller the variance of the difference, the smaller the standard error, and the higher the power. The price is having to worry about order effects, learning effects, and other things that change between the two measurements.

### Problem 20 · Unbiased vs. consistent: a counterexample for each 🗣

Give an estimator that is "unbiased but not consistent", and one that is "biased but consistent".

**Reference answer:**

**Unbiased but not consistent:** estimate the population mean using only the first observation $X_1$. $E[X_1] = \mu$, so it is unbiased; but however large the sample, its variance stays $\sigma^2$ and it never converges.

**Biased but consistent:** the variance estimator that divides by n, $\frac1n\sum(x_i - \bar x)^2$. Its expectation is $\frac{n-1}{n}\sigma^2$, so it is biased; but the bias goes to 0 as n grows, and so does the variance, so it is consistent.

This shows the two properties are independent: unbiased means "right on average", consistent means "converges to the truth as data accumulate".

## Recap

| Task | Code |
|---|---|
| CI for a mean | `x.mean() ± stats.t.ppf(0.975, n-1) * x.std(ddof=1) / np.sqrt(n)` |
| One-sample / paired / two independent samples | `ttest_1samp(x, mu)` / `ttest_rel(a, b)` / `ttest_ind(a, b, equal_var=False)` |
| Two proportions | `proportions_ztest([x1, x2], [n1, n2])` |
| Contingency table | `stats.chi2_contingency(table)`; continuity correction on by default for 2×2 |
| Nonparametric | `stats.mannwhitneyu(a, b)` |
| Bootstrap / permutation | resample with replacement, take quantiles / shuffle labels, count extreme cases |
| Power / Type I error | simulate a matrix + test with `axis=1`, look at the share significant |
| Multiple testing | `multipletests(p, method="bonferroni" / "fdr_bh")` |